# Aula 27 — Árvores de Decisão (ID3, C4.5 e CART) no SCADA Aqua Index
## Indução Automática de Regras para Qualidade e Manutenção Preditiva

---

### 1. Contexto e Contraste com a Aula 21 (BDT vs. ML)

Na **Aula 21**, modelamos a **Árvore Binária de Decisão (BDT) Contextual** para os intertravamentos da máquina pneumática de envasamento de água em copos (**Aqua Index**, Setores `000` a `500`), além da **Árvore AVL** para indexar o Historiador SCADA.

Nesta **Aula 27**, damos o passo seguinte na arquitetura de dados do SCADA: usar os registros acumulados pelo Historiador para **induzir automaticamente Árvores de Decisão** (**ID3**, **C4.5** e **CART**) voltadas a **diagnóstico de qualidade** e **manutenção preditiva**.

| Dimensão | BDT Contextual (Aula 21) | Árvores Induzidas ID3 / C4.5 / CART (Aula 27) |
| :--- | :--- | :--- |
| **Origem das regras** | Projeto determinístico manual (engenharia + FSM + regras `R-01`..`R-06`) | Indução automática a partir de dados históricos (`Historiador SCADA`) |
| **Objetivo** | Segurança operacional e intertravamento em tempo real | Classificação de qualidade do lote e alerta preditivo de degradação |
| **Complexidade** | Fixa $\mathcal{O}(h) = \mathcal{O}(1)$ com $h = 7$ | Depende da profundidade induzida (`max_depth`) |
| **Substitui segurança?** | É a própria camada de lógica segura | **NUNCA** substitui intertravamentos determinísticos nem `HS-004` *hardwired* |

> **Princípio Inegociável de Engenharia (IEC 61511 / SCADA):** Modelos estatísticos e de Aprendizado de Máquina possuem erro residual não nulo e dependem da distribuição dos dados. Eles operam exclusivamente na camada **supervisória/analítica** (IHM, KPIs, manutenção preditiva) e **jamais** substituem intertravamentos determinísticos no CLP ou o botão físico de emergência `HS-004` (*hardwired*).

> **Nota de Implementação:** Todo o núcleo algorítmico é implementado **100% do zero em Python 3.13 + NumPy**, sem `scikit-learn`. O `matplotlib` é usado apenas de forma opcional dentro de `try/except`.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import numpy as np

print(f"Ambiente carregado com sucesso | NumPy {np.__version__}")

Ambiente carregado com sucesso | NumPy 2.5.4


---
### 2. Dataset Sintético Determinístico do Historiador Aqua Index (`seed=2026`, 800 ciclos)

Os **5 atributos** são derivados diretamente das tags ISA-5.1 reais da planta (`etapa-01-logica/02 - Mapeamento_de_variaveis.md`):

1. `temp_TIT401_C` (`TIT-401`, Setor 400): Temperatura do cabeçote de termosselagem ($^\circ\text{C}$). Limiar nominal $\ge 180\,^\circ\text{C}$.
2. `tempo_prensa_E4_s` (`XV-401` / `ZSC-401`, Setor 400): Tempo de prensagem na Estação 4 (nominal $\approx 2{,}0\,\text{s}$).
3. `tempo_vacuo_PIT301_ms` (`PIT-301`, Setor 300): Tempo até confirmação de sucção da tampa no manipulador *pick-and-place* (ms).
4. `desvio_takt_SE001_ms` (`SE-001` / `M-001`, Setor 000): Desvio do tempo de indexação da mesa giratória em relação ao *takt* nominal (ms).
5. `copo_ZS102` (`ZS-102`, Setor 100): Leitura binária do sensor capacitivo de presença de copo na Estação 1 (`0` = ausente, `1` = presente).

As **4 classes operacionais** registradas após inspeção/telemetria de ciclo são:
- `0` — `CICLO_CONFORME`
- `1` — `RISCO_SELAGEM_FRACA` (cabeçote frio ou tempo de prensa insuficiente na Estação 4)
- `2` — `FALHA_TAMPA_VACUO` (lentidão/perda de vácuo em `PIT-301` na Estação 3)
- `3` — `PARADA_ABASTECIMENTO_ATRASO` (falta de copo em `ZS-102` na Estação 1 e/ou atraso severo de indexação em `SE-001`)

In [3]:
FEATURE_NAMES: List[str] = [
    "temp_TIT401_C",
    "tempo_prensa_E4_s",
    "tempo_vacuo_PIT301_ms",
    "desvio_takt_SE001_ms",
    "copo_ZS102",
]

CLASS_NAMES: List[str] = [
    "CICLO_CONFORME",
    "RISCO_SELAGEM_FRACA",
    "FALHA_TAMPA_VACUO",
    "PARADA_ABASTECIMENTO_ATRASO",
]


def gerar_dataset_aqua_index(seed: int = 2026, n_ciclos: int = 800) -> Tuple[np.ndarray, np.ndarray]:
    """Gera dataset sintético determinístico baseado nas tags ISA-5.1 da Aqua Index."""
    rng = np.random.default_rng(seed)

    # Proporções realistas de chão de fábrica (soma = 800)
    n_c0 = int(round(n_ciclos * 0.50))   # 400: CICLO_CONFORME
    n_c1 = int(round(n_ciclos * 0.19))   # 152: RISCO_SELAGEM_FRACA
    n_c2 = int(round(n_ciclos * 0.16))   # 128: FALHA_TAMPA_VACUO
    n_c3 = n_ciclos - (n_c0 + n_c1 + n_c2)  # 120: PARADA_ABASTECIMENTO_ATRASO

    # Classe 0: CICLO_CONFORME
    c0_temp = np.clip(rng.normal(186.5, 2.3, n_c0), 180.5, 195.0)
    c0_prensa = np.clip(rng.normal(2.02, 0.10, n_c0), 1.76, 2.35)
    c0_vacuo = np.clip(rng.normal(182.0, 32.0, n_c0), 85.0, 275.0)
    c0_takt = np.clip(rng.normal(10.0, 18.0, n_c0), -35.0, 68.0)
    c0_copo = np.ones(n_c0, dtype=float)
    X0 = np.column_stack([c0_temp, c0_prensa, c0_vacuo, c0_takt, c0_copo])
    y0 = np.zeros(n_c0, dtype=int)

    # Classe 1: RISCO_SELAGEM_FRACA (sub-regimes: temperatura baixa OU prensa curta)
    n_c1_temp = n_c1 // 2
    n_c1_prensa = n_c1 - n_c1_temp
    c1_temp = np.concatenate([
        np.clip(rng.normal(174.2, 3.0, n_c1_temp), 162.0, 181.5),
        np.clip(rng.normal(184.5, 2.4, n_c1_prensa), 179.0, 192.0),
    ])
    c1_prensa = np.concatenate([
        np.clip(rng.normal(1.98, 0.12, n_c1_temp), 1.65, 2.30),
        np.clip(rng.normal(1.46, 0.14, n_c1_prensa), 1.05, 1.78),
    ])
    c1_vacuo = np.clip(rng.normal(190.0, 34.0, n_c1), 90.0, 285.0)
    c1_takt = np.clip(rng.normal(16.0, 20.0, n_c1), -30.0, 75.0)
    c1_copo = np.ones(n_c1, dtype=float)
    X1 = np.column_stack([c1_temp, c1_prensa, c1_vacuo, c1_takt, c1_copo])
    y1 = np.ones(n_c1, dtype=int)

    # Classe 2: FALHA_TAMPA_VACUO (pressostato PIT-301 lento ou sem confirmação)
    c2_temp = np.clip(rng.normal(185.5, 2.6, n_c2), 178.0, 194.0)
    c2_prensa = np.clip(rng.normal(2.00, 0.12, n_c2), 1.68, 2.32)
    c2_vacuo = np.clip(rng.normal(385.0, 58.0, n_c2), 260.0, 560.0)
    c2_takt = np.clip(rng.normal(28.0, 24.0, n_c2), -25.0, 95.0)
    c2_copo = np.ones(n_c2, dtype=float)
    X2 = np.column_stack([c2_temp, c2_prensa, c2_vacuo, c2_takt, c2_copo])
    y2 = np.full(n_c2, 2, dtype=int)

    # Classe 3: PARADA_ABASTECIMENTO_ATRASO (copo ausente em ZS-102 ou atraso severo em SE-001)
    n_c3_sem_copo = int(round(n_c3 * 0.65))
    n_c3_atraso = n_c3 - n_c3_sem_copo
    c3_temp = np.clip(rng.normal(184.0, 3.0, n_c3), 175.0, 193.0)
    c3_prensa = np.clip(rng.normal(1.96, 0.14, n_c3), 1.55, 2.30)
    c3_vacuo = np.clip(rng.normal(205.0, 40.0, n_c3), 100.0, 320.0)
    c3_takt = np.concatenate([
        np.clip(rng.normal(88.0, 28.0, n_c3_sem_copo), 25.0, 180.0),
        np.clip(rng.normal(148.0, 30.0, n_c3_atraso), 82.0, 240.0),
    ])
    c3_copo = np.concatenate([
        np.zeros(n_c3_sem_copo, dtype=float),
        np.ones(n_c3_atraso, dtype=float),
    ])
    X3 = np.column_stack([c3_temp, c3_prensa, c3_vacuo, c3_takt, c3_copo])
    y3 = np.full(n_c3, 3, dtype=int)

    X = np.vstack([X0, X1, X2, X3])
    y = np.concatenate([y0, y1, y2, y3])

    perm = rng.permutation(len(y))
    return X[perm], y[perm]


def split_estratificado(
    X: np.ndarray, y: np.ndarray, test_ratio: float = 0.30, seed: int = 2026
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Divide X e y de forma estratificada e determinística (sem scikit-learn)."""
    rng = np.random.default_rng(seed)
    train_idx: List[int] = []
    test_idx: List[int] = []

    for cls in np.unique(y):
        idx_cls = np.where(y == cls)[0]
        rng.shuffle(idx_cls)
        n_test = int(round(len(idx_cls) * test_ratio))
        test_idx.extend(idx_cls[:n_test].tolist())
        train_idx.extend(idx_cls[n_test:].tolist())

    train_arr = np.array(train_idx, dtype=int)
    test_arr = np.array(test_idx, dtype=int)
    rng.shuffle(train_arr)
    rng.shuffle(test_arr)
    return X[train_arr], X[test_arr], y[train_arr], y[test_arr]


X, y = gerar_dataset_aqua_index(seed=2026, n_ciclos=800)
X_train, X_test, y_train, y_test = split_estratificado(X, y, test_ratio=0.30, seed=2026)

assert X.shape == (800, 5), f"Formato inesperado: {X.shape}"
assert len(X_train) == 560 and len(X_test) == 240, "Split 70/30 deve gerar 560/240 amostras"
assert set(np.unique(y_train)) == {0, 1, 2, 3}
assert set(np.unique(y_test)) == {0, 1, 2, 3}

print(f"Dataset gerado: X={X.shape}, Treino={X_train.shape}, Teste={X_test.shape}")
for cls_id, nome in enumerate(CLASS_NAMES):
    tot = int(np.sum(y == cls_id))
    tr = int(np.sum(y_train == cls_id))
    te = int(np.sum(y_test == cls_id))
    print(f"  Classe {cls_id} ({nome:<28}): total={tot:3d} | treino={tr:3d} | teste={te:3d}")

Dataset gerado: X=(800, 5), Treino=(560, 5), Teste=(240, 5)
  Classe 0 (CICLO_CONFORME              ): total=400 | treino=280 | teste=120
  Classe 1 (RISCO_SELAGEM_FRACA         ): total=152 | treino=106 | teste= 46
  Classe 2 (FALHA_TAMPA_VACUO           ): total=128 | treino= 90 | teste= 38
  Classe 3 (PARADA_ABASTECIMENTO_ATRASO ): total=120 | treino= 84 | teste= 36


---
### 3. Funções de Impureza e Critérios de Particionamento (ID3, C4.5 e CART)

Dado um conjunto de rótulos $S$ com proporções de classe $p_k = \frac{|S_k|}{|S|}$:

- **Entropia de Shannon (ID3 / C4.5):**
  $$H(S) = -\sum_{k=1}^{C} p_k \log_2(p_k)$$
- **Impureza de Gini (CART):**
  $$G(S) = 1 - \sum_{k=1}^{C} p_k^2$$
- **Ganho de Informação (ID3):**
  $$IG(S, t) = H(S) - \left(\frac{|S_{\text{esq}}|}{|S|} H(S_{\text{esq}}) + \frac{|S_{\text{dir}}|}{|S|} H(S_{\text{dir}})\right)$$
- **Razão de Ganho (C4.5):**
  $$GR(S, t) = \frac{IG(S, t)}{\text{SplitInfo}(S, t)}, \quad \text{SplitInfo}(S, t) = -\sum_{j \in \{\text{esq}, \text{dir}\}} \frac{|S_j|}{|S|}\log_2\left(\frac{|S_j|}{|S|}\right)$$

In [5]:
def entropia(y: np.ndarray) -> float:
    """Calcula a Entropia de Shannon (base 2) de um vetor de rótulos."""
    n = len(y)
    if n == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    probs = counts / n
    probs = probs[probs > 0.0]
    return float(-np.sum(probs * np.log2(probs)))


def gini(y: np.ndarray) -> float:
    """Calcula o índice de impureza de Gini de um vetor de rótulos."""
    n = len(y)
    if n == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    probs = counts / n
    return float(1.0 - np.sum(probs ** 2))


def ganho_informacao(y: np.ndarray, y_esq: np.ndarray, y_dir: np.ndarray) -> float:
    """Calcula o Ganho de Informação (ID3) de uma partição binária."""
    n = len(y)
    if n == 0 or len(y_esq) == 0 or len(y_dir) == 0:
        return 0.0
    h_pai = entropia(y)
    h_filhos = (len(y_esq) / n) * entropia(y_esq) + (len(y_dir) / n) * entropia(y_dir)
    return float(h_pai - h_filhos)


def razao_ganho(y: np.ndarray, y_esq: np.ndarray, y_dir: np.ndarray) -> float:
    """Calcula a Razão de Ganho (C4.5), normalizando o Ganho de Informação pela SplitInfo."""
    n = len(y)
    if n == 0 or len(y_esq) == 0 or len(y_dir) == 0:
        return 0.0
    ig = ganho_informacao(y, y_esq, y_dir)
    p_esq = len(y_esq) / n
    p_dir = len(y_dir) / n
    split_info = -(p_esq * np.log2(p_esq) + p_dir * np.log2(p_dir))
    if split_info <= 1e-12:
        return 0.0
    return float(ig / split_info)


# Testes unitários das funções matemáticas de impureza
assert abs(entropia(np.array([0, 0, 0, 0])) - 0.0) < 1e-12
assert abs(gini(np.array([0, 0, 0, 0])) - 0.0) < 1e-12
assert abs(entropia(np.array([0, 1])) - 1.0) < 1e-12
assert abs(gini(np.array([0, 1])) - 0.5) < 1e-12
assert abs(entropia(np.array([0, 1, 2, 3])) - 2.0) < 1e-12
assert abs(gini(np.array([0, 1, 2, 3])) - 0.75) < 1e-12
assert abs(ganho_informacao(np.array([0, 0, 1, 1]), np.array([0, 0]), np.array([1, 1])) - 1.0) < 1e-12
assert abs(razao_ganho(np.array([0, 0, 1, 1]), np.array([0, 0]), np.array([1, 1])) - 1.0) < 1e-12
assert razao_ganho(np.array([0, 1]), np.array([0, 1]), np.array([])) == 0.0

print("Testes unitários de entropia, gini, ganho_informacao e razao_ganho aprovados!")

Testes unitários de entropia, gini, ganho_informacao e razao_ganho aprovados!


---
### 4. Implementação do Zero: `ArvoreDecisaoClassica` e Extração de Regras

A classe `ArvoreDecisaoClassica` suporta:
- `criterion="gini"` (redução ponderada da impureza de Gini — abordagem **CART**)
- `criterion="entropy"` (maximização do Ganho de Informação via Entropia — abordagem **ID3/C4.5** com limiares contínuos)
- Critérios de parada: `max_depth`, `min_samples_split` e pureza de folha
- Extração automática de regras legíveis `SE ... ENTÃO ...` com suporte e confiança da folha.

In [7]:
@dataclass
class NoArvore:
    """Nó interno ou folha da Árvore de Decisão."""
    feature_idx: Optional[int] = None
    threshold: Optional[float] = None
    esquerda: Optional["NoArvore"] = None
    direita: Optional["NoArvore"] = None
    classe_predita: Optional[int] = None
    contagens_classe: Optional[np.ndarray] = None
    impureza: float = 0.0
    n_amostras: int = 0

    @property
    def eh_folha(self) -> bool:
        return self.classe_predita is not None


class ArvoreDecisaoClassica:
    """Árvore de Decisão Clássica implementada 100% do zero com NumPy."""

    def __init__(
        self,
        criterion: str = "gini",
        max_depth: int = 5,
        min_samples_split: int = 5,
    ) -> None:
        if criterion not in ("gini", "entropy"):
            raise ValueError(f"Critério inválido: {criterion!r}. Use 'gini' ou 'entropy'.")
        if max_depth < 1:
            raise ValueError("max_depth deve ser >= 1.")
        if min_samples_split < 2:
            raise ValueError("min_samples_split deve ser >= 2.")

        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.raiz: Optional[NoArvore] = None
        self.n_classes_: int = 0
        self.n_features_: int = 0

    def _impureza(self, y: np.ndarray) -> float:
        return gini(y) if self.criterion == "gini" else entropia(y)

    def _ganho_particao(self, y: np.ndarray, y_esq: np.ndarray, y_dir: np.ndarray) -> float:
        n = len(y)
        if len(y_esq) == 0 or len(y_dir) == 0:
            return 0.0
        imp_pai = self._impureza(y)
        imp_filhos = (len(y_esq) / n) * self._impureza(y_esq) + (len(y_dir) / n) * self._impureza(y_dir)
        return float(imp_pai - imp_filhos)

    def _melhor_divisao(self, X: np.ndarray, y: np.ndarray) -> Tuple[Optional[int], Optional[float], float]:
        melhor_feat: Optional[int] = None
        melhor_thresh: Optional[float] = None
        melhor_ganho = 0.0

        for feat_idx in range(self.n_features_):
            valores = np.unique(X[:, feat_idx])
            if len(valores) <= 1:
                continue
            limiares = 0.5 * (valores[:-1] + valores[1:])
            col = X[:, feat_idx]
            for thresh in limiares:
                mask_esq = col <= thresh
                if not np.any(mask_esq) or np.all(mask_esq):
                    continue
                ganho = self._ganho_particao(y, y[mask_esq], y[~mask_esq])
                if ganho > melhor_ganho:
                    melhor_ganho = ganho
                    melhor_feat = feat_idx
                    melhor_thresh = float(thresh)

        return melhor_feat, melhor_thresh, melhor_ganho

    def _contar_classes(self, y: np.ndarray) -> np.ndarray:
        return np.bincount(y, minlength=self.n_classes_)

    def _construir_no(self, X: np.ndarray, y: np.ndarray, profundidade: int) -> NoArvore:
        contagens = self._contar_classes(y)
        classe_majoritaria = int(np.argmax(contagens))
        imp = self._impureza(y)
        n_amostras = len(y)

        # Condições de parada (folha)
        if (
            profundidade >= self.max_depth
            or n_amostras < self.min_samples_split
            or len(np.unique(y)) == 1
        ):
            return NoArvore(
                classe_predita=classe_majoritaria,
                contagens_classe=contagens,
                impureza=imp,
                n_amostras=n_amostras,
            )

        feat_idx, thresh, ganho = self._melhor_divisao(X, y)
        if feat_idx is None or thresh is None or ganho <= 1e-12:
            return NoArvore(
                classe_predita=classe_majoritaria,
                contagens_classe=contagens,
                impureza=imp,
                n_amostras=n_amostras,
            )

        mask_esq = X[:, feat_idx] <= thresh
        filho_esq = self._construir_no(X[mask_esq], y[mask_esq], profundidade + 1)
        filho_dir = self._construir_no(X[~mask_esq], y[~mask_esq], profundidade + 1)

        return NoArvore(
            feature_idx=feat_idx,
            threshold=thresh,
            esquerda=filho_esq,
            direita=filho_dir,
            contagens_classe=contagens,
            impureza=imp,
            n_amostras=n_amostras,
        )

    def fit(self, X: np.ndarray, y: np.ndarray) -> "ArvoreDecisaoClassica":
        """Ajusta a árvore aos dados de treino (X, y)."""
        X_arr = np.asarray(X, dtype=float)
        y_arr = np.asarray(y, dtype=int)
        self.n_features_ = X_arr.shape[1]
        self.n_classes_ = int(np.max(y_arr)) + 1
        self.raiz = self._construir_no(X_arr, y_arr, profundidade=0)
        return self

    def _prever_amostra(self, x: np.ndarray, no: NoArvore) -> int:
        while not no.eh_folha:
            assert no.feature_idx is not None and no.threshold is not None
            if x[no.feature_idx] <= no.threshold:
                assert no.esquerda is not None
                no = no.esquerda
            else:
                assert no.direita is not None
                no = no.direita
        assert no.classe_predita is not None
        return no.classe_predita

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Prediz os rótulos de classe para cada linha de X."""
        if self.raiz is None:
            raise RuntimeError("A árvore ainda não foi treinada. Chame fit() antes de predict().")
        X_arr = np.asarray(X, dtype=float)
        return np.array([self._prever_amostra(x, self.raiz) for x in X_arr], dtype=int)

    def profundidade_efetiva(self) -> int:
        """Retorna a altura efetiva da árvore induzida."""
        def _alt(no: Optional[NoArvore]) -> int:
            if no is None or no.eh_folha:
                return 0
            return 1 + max(_alt(no.esquerda), _alt(no.direita))
        return _alt(self.raiz)

    def contar_nos_e_folhas(self) -> Tuple[int, int]:
        """Retorna (total_nos, total_folhas)."""
        def _contar(no: Optional[NoArvore]) -> Tuple[int, int]:
            if no is None:
                return 0, 0
            if no.eh_folha:
                return 1, 1
            n_e, f_e = _contar(no.esquerda)
            n_d, f_d = _contar(no.direita)
            return 1 + n_e + n_d, f_e + f_d
        return _contar(self.raiz)

    def extrair_regras(self, feature_names: List[str], class_names: List[str]) -> List[str]:
        """Extrai todas as regras legíveis da raiz às folhas no formato SE ... ENTÃO ...."""
        regras: List[str] = []

        def _dfs(no: Optional[NoArvore], condicoes: List[str]) -> None:
            if no is None:
                return
            if no.eh_folha:
                assert no.classe_predita is not None and no.contagens_classe is not None
                suporte = no.n_amostras
                conf = (no.contagens_classe[no.classe_predita] / suporte) * 100.0 if suporte > 0 else 0.0
                ant = " E ".join(condicoes) if condicoes else "VERDADEIRO"
                regras.append(
                    f"SE {ant} ENTÃO {class_names[no.classe_predita]} "
                    f"(suporte={suporte}, confiança={conf:.1f}%)"
                )
                return
            assert no.feature_idx is not None and no.threshold is not None
            nome = feature_names[no.feature_idx]
            _dfs(no.esquerda, condicoes + [f"{nome} <= {no.threshold:.2f}"])
            _dfs(no.direita, condicoes + [f"{nome} > {no.threshold:.2f}"])

        _dfs(self.raiz, [])
        return regras


print("Classe ArvoreDecisaoClassica definida com sucesso.")

Classe ArvoreDecisaoClassica definida com sucesso.


---
### 5. Métricas de Avaliação Implementadas do Zero (Matriz 4×4 e Macro-Métricas)

Avaliamos ambos os critérios (`gini` e `entropy`) no conjunto de teste estratificado ($240$ ciclos) utilizando exclusivamente funções implementadas com NumPy.

In [9]:
def matriz_confusao(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 4) -> np.ndarray:
    '''Constrói a matriz de confusão n_classes x n_classes (linhas=real, colunas=predito).'''
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for yt, yp in zip(y_true, y_pred):
        cm[int(yt), int(yp)] += 1
    return cm


def metricas_classificacao(cm: np.ndarray) -> Dict[str, object]:
    '''Calcula Acurácia, Precisão/Recall/F1 por classe e médias macro a partir da matriz de confusão.'''
    total = int(np.sum(cm))
    acertos = int(np.trace(cm))
    acuracia = acertos / total if total > 0 else 0.0

    tp = np.diag(cm).astype(float)
    fp = np.sum(cm, axis=0).astype(float) - tp
    fn = np.sum(cm, axis=1).astype(float) - tp

    precisao_cls = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    recall_cls = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    denom_f1 = precisao_cls + recall_cls
    f1_cls = np.divide(
        2.0 * precisao_cls * recall_cls,
        denom_f1,
        out=np.zeros_like(tp),
        where=denom_f1 > 0,
    )

    return {
        "acuracia": float(acuracia),
        "precisao_macro": float(np.mean(precisao_cls)),
        "recall_macro": float(np.mean(recall_cls)),
        "f1_macro": float(np.mean(f1_cls)),
        "precisao_classe": precisao_cls,
        "recall_classe": recall_cls,
        "f1_classe": f1_cls,
    }


def exibir_relatorio(nome_modelo: str, y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[np.ndarray, Dict[str, object]]:
    cm = matriz_confusao(y_true, y_pred, n_classes=len(CLASS_NAMES))
    met = metricas_classificacao(cm)
    print(f"=== {nome_modelo} ===")
    print(
        f"Acurácia: {met['acuracia']*100:.2f}% | "
        f"Precisão macro: {met['precisao_macro']*100:.2f}% | "
        f"Recall macro: {met['recall_macro']*100:.2f}% | "
        f"F1 macro: {met['f1_macro']*100:.2f}%"
    )
    print("\nMatriz de Confusão 4x4 (linhas=Real, colunas=Predito):")
    header = "Real \\ Pred".ljust(30) + " ".join(f"C{i}".rjust(6) for i in range(4))
    print(header)
    for i, cname in enumerate(CLASS_NAMES):
        row_vals = " ".join(f"{cm[i, j]:6d}" for j in range(4))
        print(f"C{i} {cname:<27}{row_vals}")

    print("\nMétricas por Classe:")
    for i, cname in enumerate(CLASS_NAMES):
        p = met["precisao_classe"][i] * 100.0
        r = met["recall_classe"][i] * 100.0
        f1 = met["f1_classe"][i] * 100.0
        print(f"  C{i} {cname:<28} -> P={p:6.2f}% | R={r:6.2f}% | F1={f1:6.2f}%")
    print()
    return cm, met


modelo_gini = ArvoreDecisaoClassica(criterion="gini", max_depth=5, min_samples_split=5).fit(X_train, y_train)
modelo_entropia = ArvoreDecisaoClassica(criterion="entropy", max_depth=5, min_samples_split=5).fit(X_train, y_train)

pred_gini = modelo_gini.predict(X_test)
pred_entropia = modelo_entropia.predict(X_test)

cm_gini, met_gini = exibir_relatorio("CART (criterion='gini', max_depth=5)", y_test, pred_gini)
cm_ent, met_ent = exibir_relatorio("ID3/C4.5 (criterion='entropy', max_depth=5)", y_test, pred_entropia)

# Validações obrigatórias
assert cm_gini.shape == (4, 4) and cm_ent.shape == (4, 4)
assert int(np.sum(cm_gini)) == len(y_test) == 240
assert int(np.sum(cm_ent)) == len(y_test) == 240
assert int(np.trace(cm_gini)) == int(np.sum(pred_gini == y_test))
assert int(np.trace(cm_ent)) == int(np.sum(pred_entropia == y_test))
assert met_gini["acuracia"] > 0.90, f"Acurácia Gini abaixo de 90%: {met_gini['acuracia']:.4f}"
assert met_ent["acuracia"] > 0.90, f"Acurácia Entropia abaixo de 90%: {met_ent['acuracia']:.4f}"

print("Asserts de acurácia (> 90%) e consistência da matriz de confusão 4x4 aprovados!")

=== CART (criterion='gini', max_depth=5) ===
Acurácia: 97.08% | Precisão macro: 95.99% | Recall macro: 95.96% | F1 macro: 95.97%

Matriz de Confusão 4x4 (linhas=Real, colunas=Predito):
Real \ Pred                       C0     C1     C2     C3
C0 CICLO_CONFORME                119      0      1      0
C1 RISCO_SELAGEM_FRACA             1     44      0      1
C2 FALHA_TAMPA_VACUO               0      0     37      1
C3 PARADA_ABASTECIMENTO_ATRASO     0      2      1     33

Métricas por Classe:
  C0 CICLO_CONFORME               -> P= 99.17% | R= 99.17% | F1= 99.17%
  C1 RISCO_SELAGEM_FRACA          -> P= 95.65% | R= 95.65% | F1= 95.65%
  C2 FALHA_TAMPA_VACUO            -> P= 94.87% | R= 97.37% | F1= 96.10%
  C3 PARADA_ABASTECIMENTO_ATRASO  -> P= 94.29% | R= 91.67% | F1= 92.96%

=== ID3/C4.5 (criterion='entropy', max_depth=5) ===
Acurácia: 97.50% | Precisão macro: 96.57% | Recall macro: 96.66% | F1 macro: 96.60%

Matriz de Confusão 4x4 (linhas=Real, colunas=Predito):
Real \ Pred           

In [10]:
# Análise do nó raiz: Ganho de Informação (ID3) vs Razão de Ganho (C4.5) vs Redução de Gini (CART)
print("Comparativo de critérios no Nó Raiz (560 ciclos de treino):")
print(f"{'Atributo':<24} {'Limiar Ótimo':>12} {'IG (ID3)':>10} {'GainRatio (C4.5)':>18} {'ΔGini (CART)':>14}")
print("-" * 82)

g_pai = gini(y_train)
for idx, fname in enumerate(FEATURE_NAMES):
    vals = np.unique(X_train[:, idx])
    limiares = 0.5 * (vals[:-1] + vals[1:])
    melhor_ig, melhor_gr, melhor_dg = 0.0, 0.0, 0.0
    t_ig = 0.0
    col = X_train[:, idx]
    for t in limiares:
        m = col <= t
        ig = ganho_informacao(y_train, y_train[m], y_train[~m])
        gr = razao_ganho(y_train, y_train[m], y_train[~m])
        dg = g_pai - ((np.sum(m) / len(y_train)) * gini(y_train[m]) + (np.sum(~m) / len(y_train)) * gini(y_train[~m]))
        if ig > melhor_ig:
            melhor_ig = ig
            t_ig = float(t)
        if gr > melhor_gr:
            melhor_gr = gr
        if dg > melhor_dg:
            melhor_dg = dg
    print(f"{fname:<24} {t_ig:12.2f} {melhor_ig:10.4f} {melhor_gr:18.4f} {melhor_dg:14.4f}")

print("\nRegras extraídas (CART — criterion='gini', max_depth=5):")
regras_gini = modelo_gini.extrair_regras(FEATURE_NAMES, CLASS_NAMES)
assert len(regras_gini) > 0
for i, regra in enumerate(regras_gini, start=1):
    print(f"  R{i:02d}: {regra}")

Comparativo de critérios no Nó Raiz (560 ciclos de treino):
Atributo                 Limiar Ótimo   IG (ID3)   GainRatio (C4.5)   ΔGini (CART)
----------------------------------------------------------------------------------
temp_TIT401_C                  180.44     0.2481             0.5041         0.0946
tempo_prensa_E4_s                1.76     0.2391             0.5335         0.0900
tempo_vacuo_PIT301_ms          259.90     0.6201             0.9622         0.1908
desvio_takt_SE001_ms            62.44     0.5012             0.8407         0.1612
copo_ZS102                       0.50     0.3463             0.7212         0.1195

Regras extraídas (CART — criterion='gini', max_depth=5):
  R01: SE tempo_vacuo_PIT301_ms <= 259.90 E desvio_takt_SE001_ms <= 66.24 E temp_TIT401_C <= 180.40 ENTÃO RISCO_SELAGEM_FRACA (suporte=58, confiança=100.0%)
  R02: SE tempo_vacuo_PIT301_ms <= 259.90 E desvio_takt_SE001_ms <= 66.24 E temp_TIT401_C > 180.40 E tempo_prensa_E4_s <= 1.74 ENTÃO RISCO_SELAG

---
### 6. Curva de Complexidade (`max_depth = 1..8`): Subajuste (*Underfitting*) vs. Sobreajuste (*Overfitting*)

Variamos `max_depth` de `1` a `8` para observar a evolução da acurácia em treino e teste, bem como o crescimento estrutural (nós e folhas) em ambos os critérios.

In [12]:
profundidades = list(range(1, 9))
historico: Dict[str, List[Dict[str, float]]] = {"gini": [], "entropy": []}

print(
    f"{'Critério':<9} {'Prof.':>5} {'Acc Treino':>11} {'Acc Teste':>11} "
    f"{'F1 Teste':>10} {'Nós':>6} {'Folhas':>7}"
)
print("-" * 66)

for crit in ("gini", "entropy"):
    for d in profundidades:
        arv = ArvoreDecisaoClassica(criterion=crit, max_depth=d, min_samples_split=5).fit(X_train, y_train)
        acc_tr = float(np.mean(arv.predict(X_train) == y_train))
        pred_te = arv.predict(X_test)
        cm_te = matriz_confusao(y_test, pred_te, n_classes=4)
        met_te = metricas_classificacao(cm_te)
        n_nos, n_folhas = arv.contar_nos_e_folhas()

        historico[crit].append({
            "depth": d,
            "acc_train": acc_tr,
            "acc_test": float(met_te["acuracia"]),
            "f1_test": float(met_te["f1_macro"]),
            "nos": n_nos,
            "folhas": n_folhas,
        })
        print(
            f"{crit:<9} {d:5d} {acc_tr*100:10.2f}% {met_te['acuracia']*100:10.2f}% "
            f"{met_te['f1_macro']*100:9.2f}% {n_nos:6d} {n_folhas:7d}"
        )

# Asserts de consistência da curva
for crit in ("gini", "entropy"):
    accs_tr = [r["acc_train"] for r in historico[crit]]
    accs_te = [r["acc_test"] for r in historico[crit]]
    assert accs_tr[-1] >= accs_tr[0], f"Acurácia treino não evoluiu em {crit}"
    assert max(accs_te) > 0.90, f"Melhor acurácia teste <= 90% em {crit}"

print("\nCurva max_depth = 1..8 validada com sucesso.")

# Gráfico opcional via matplotlib
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    for crit, label, color in [("gini", "CART (Gini)", "#0275d8"), ("entropy", "ID3/C4.5 (Entropia)", "#d9534f")]:
        ds = [r["depth"] for r in historico[crit]]
        tr = [r["acc_train"] * 100 for r in historico[crit]]
        te = [r["acc_test"] * 100 for r in historico[crit]]
        axes[0].plot(ds, tr, linestyle="--", marker="o", alpha=0.75, color=color, label=f"{label} — Treino")
        axes[0].plot(ds, te, linestyle="-", marker="s", linewidth=2, color=color, label=f"{label} — Teste")

    axes[0].set_title("Curva Treino vs. Teste (max_depth = 1..8)")
    axes[0].set_xlabel("Profundidade Máxima (max_depth)")
    axes[0].set_ylabel("Acurácia (%)")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=8)

    im = axes[1].imshow(cm_gini, cmap="Blues")
    axes[1].set_title("Matriz de Confusão 4x4 — CART (Gini, depth=5)")
    axes[1].set_xlabel("Classe Predita")
    axes[1].set_ylabel("Classe Real")
    axes[1].set_xticks(range(4))
    axes[1].set_yticks(range(4))
    for i in range(4):
        for j in range(4):
            axes[1].text(j, i, str(cm_gini[i, j]), ha="center", va="center", color="black")
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()
    print("Gráficos renderizados com sucesso via matplotlib.")
except ImportError:
    print("matplotlib não está instalado neste ambiente — gráficos ignorados (núcleo algorítmico 100% validado).")

Critério  Prof.  Acc Treino   Acc Teste   F1 Teste    Nós  Folhas
------------------------------------------------------------------
gini          1      66.07%      65.42%     42.00%      3       2
gini          2      79.64%      77.92%     67.15%      7       4
gini          3      90.00%      87.92%     85.87%     11       6
gini          4      98.21%      96.25%     94.99%     13       7
gini          5      99.29%      97.08%     95.97%     15       8
gini          6      99.29%      97.08%     95.97%     17       9
gini          7      99.64%      97.08%     95.97%     19      10
gini          8      99.64%      97.08%     95.97%     19      10
entropy       1      66.07%      65.42%     42.00%      3       2
entropy       2      79.64%      78.33%     67.61%      7       4
entropy       3      90.00%      88.33%     86.49%     11       6
entropy       4      98.57%      96.67%     95.65%     15       8
entropy       5      99.46%      97.50%     96.60%     17       9
entropy  